# Reviewer entry point: forward transfer

Use the same working directory as Notebook 02. Materialize the benchmark
harness and fetch the pinned sources using Notebook 02, then restore the
original `preflight_status.json` and validation-selected RGB `best.pt` files
under `mahimandal160_25mcsa19/outputs/mahimandal160_paper_benchmark_v1/`.
Reuse the frozen Mahimandal release in that benchmark workspace. Place the
official OLI2MSI TIFF folders or `oli2msi-thesis.zip` under
`mahimandal_oli_transfer_25mcsa19/datasets/oli2msi/`. The notebook checks the
official 5,225 training and 100 test pairs before evaluating.

This submission copy removes the separate architecture-search setup and
evaluation from the original transfer notebook. The published baseline
inference, metric and aggregation code is retained; results go to the separate
`notebook3_baselines` directory.


# Mahimandal-trained baselines on OLI2MSI

Evaluate the seven published RGB implementations and bicubic on the 100
official OLI2MSI test pairs. Use the validation-selected Mahimandal seed-42
checkpoints, full frames, linear reflectance and a three-pixel HR border crop.
The source checkpoint label is `reflectance16`; the target OLI2MSI arrays
retain their supplied float32 encoding.


## 1. Reuse the owned DGX workspaces


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
import pandas as pd
import torch
from IPython.display import display, FileLink

if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError('Run with exactly one scheduler-isolated DGX GPU')

SESSION = Path('.').resolve()
if SESSION.name.casefold() != '25mcsa19':
    raise RuntimeError('Start Jupyter from your 25mcsa19 directory')
BENCH_HOME = SESSION / 'mahimandal160_25mcsa19'
CROSS_HOME = SESSION / 'mahimandal_oli_transfer_25mcsa19'
BENCH_HARNESS = BENCH_HOME / 'benchmark_harness' / 'mahimandal160'
BENCH_SOURCES = BENCH_HOME / 'official_sources' / 'mahimandal160'
BENCH_RESULTS = BENCH_HOME / 'outputs' / 'mahimandal160_paper_benchmark_v1'
LOCAL_SITE = BENCH_HOME / '.python-packages' / 'site-packages'
for required in (BENCH_HARNESS / 'adapters.py', BENCH_SOURCES,
                 BENCH_RESULTS / 'preflight_status.json'):
    if not required.exists():
        raise FileNotFoundError(f'Prepare the seed-42 benchmark workspace: {required}')
CROSS_HOME.mkdir(parents=True, exist_ok=True)
CODE_DIR = CROSS_HOME / 'code_notebook3_baselines'
CODE_DIR.mkdir(exist_ok=True)
PYTHON = sys.executable
OWNED_PYTHONPATH = os.pathsep.join(map(str, (
    CODE_DIR, BENCH_HARNESS, LOCAL_SITE
)))
for folder in (CODE_DIR, BENCH_HARNESS, LOCAL_SITE):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

def run(command):
    command = list(map(str, command))
    env = os.environ.copy()
    env['PYTHONPATH'] = OWNED_PYTHONPATH
    env['PYTHONUNBUFFERED'] = '1'
    env['KAGGLE_CONFIG_DIR'] = str(BENCH_HOME / 'secrets' / 'kaggle')
    print('+', ' '.join(command), flush=True)
    return subprocess.run(command, env=env, check=True)

mah_candidates = sorted((BENCH_HOME / 'datasets' / 'mahimandal160').rglob(
    'paper_release_manifest.csv'))
mah_candidates = [p.parent for p in mah_candidates if (p.parent / 'test' / 'LR').is_dir()]
if len(mah_candidates) != 1:
    raise RuntimeError(f'Expected one Mahimandal release: {mah_candidates}')
MAH_ROOT = mah_candidates[0]
print('Mahimandal:', MAH_ROOT)


## 2. Locate or download the official OLI2MSI TIFF pairs


In [ ]:
OLI_CONTAINER = CROSS_HOME / 'datasets' / 'oli2msi'
OLI_CONTAINER.mkdir(parents=True, exist_ok=True)

def oli_roots(container):
    roots = []
    for path in container.rglob('train_lr'):
        if path.is_dir() and all((path.parent / part).is_dir() for part in
                                 ('train_hr', 'test_lr', 'test_hr')):
            roots.append(path.parent)
    return sorted(set(roots))

roots = oli_roots(OLI_CONTAINER)
if not roots:
    archive = OLI_CONTAINER / 'oli2msi-thesis.zip'
    if not archive.is_file():
        credentials = BENCH_HOME / 'secrets' / 'kaggle' / 'kaggle.json'
        if not credentials.is_file():
            raise FileNotFoundError(
                f'Place kaggle.json at {credentials} or the OLI2MSI dataset ZIP at {archive}'
            )
        run([PYTHON, '-c', 'from kaggle.cli import main; main()',
             'datasets', 'download', '-d', 'twilight2002/oli2msi-thesis',
             '-p', OLI_CONTAINER])
    if not zipfile.is_zipfile(archive):
        raise ValueError(f'Incomplete OLI2MSI ZIP: {archive}')
    marker = OLI_CONTAINER / '.extracted'
    sha = hashlib.sha256()
    with archive.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            sha.update(chunk)
    archive_sha = sha.hexdigest()
    if not marker.is_file() or marker.read_text().strip() != archive_sha:
        with zipfile.ZipFile(archive) as source:
            base = OLI_CONTAINER.resolve()
            for member in source.infolist():
                target = (OLI_CONTAINER / member.filename).resolve()
                try:
                    target.relative_to(base)
                except ValueError:
                    raise ValueError(f'Unsafe ZIP member: {member.filename}')
            source.extractall(OLI_CONTAINER)
        marker.write_text(archive_sha)
    roots = oli_roots(OLI_CONTAINER)
if len(roots) != 1:
    raise RuntimeError(f'Expected one OLI2MSI root, found {roots}')
OLI_ROOT = roots[0]
print('OLI2MSI:', OLI_ROOT)


## 3. Lock the transfer runner and inspect bands


In [ ]:
RUNNER_SOURCE = '"""Full-frame transfer study for the pinned benchmark and CrossSensorNAS models.\n\nThe source checkpoints stay immutable. OLI2MSI validation is derived only from\nits official training directory; the official 100-pair test directory is locked.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport rasterio\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom adapters import load as load_benchmark, load_discriminator, predict as predict_benchmark\nfrom data import Pairs as MahimandalPairs, digest\nfrom metrics import mae, metric_bundle, psnr\nPROFILES = ("rgb", "rgbnir")\n\n\ndef write_json(path: Path, value: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef write_csv(path: Path, rows: list[dict]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(dict.fromkeys(k for r in rows for k in r)))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef oli_layout(root: Path) -> dict[str, Path]:\n    aliases = {"trainlr": "train_lr", "trainhr": "train_hr",\n               "testlr": "test_lr", "testhr": "test_hr"}\n    found = {}\n    for child in root.iterdir():\n        if child.is_dir():\n            name = "".join(c for c in child.name.casefold() if c.isalnum())\n            if name in aliases:\n                found[aliases[name]] = child\n    if set(found) != set(aliases.values()):\n        raise ValueError(f"Expected train_lr/train_hr/test_lr/test_hr under {root}")\n    return found\n\n\ndef oli_inventory(root: Path, seed: int = 42, val_percent: int = 10) -> dict:\n    layout = oli_layout(root)\n    inventory = {}\n    manifest = hashlib.sha256()\n    for split in ("train", "test"):\n        sides = {}\n        for side in ("lr", "hr"):\n            paths = [p for p in layout[f"{split}_{side}"].iterdir()\n                     if p.is_file() and p.suffix.casefold() in {".tif", ".tiff"}]\n            files = {p.stem.casefold(): p for p in paths}\n            if len(files) != len(paths):\n                raise ValueError(f"Duplicate pair identifiers in {layout[split + \'_\' + side]}")\n            sides[side] = files\n        if not sides["lr"] or sides["lr"].keys() != sides["hr"].keys():\n            raise ValueError(f"Unpaired or empty OLI2MSI {split} files")\n        inventory[split] = []\n        for name in sorted(sides["lr"]):\n            lr, hr = sides["lr"][name], sides["hr"][name]\n            manifest.update(f"{split}|{name}|{lr.stat().st_size}|{hr.stat().st_size}\\n".encode())\n            inventory[split].append((name, lr, hr))\n    if {row[0] for row in inventory["train"]} & {row[0] for row in inventory["test"]}:\n        raise ValueError("OLI2MSI train and test share pair identifiers")\n    with rasterio.open(inventory["train"][0][1]) as lr, rasterio.open(inventory["train"][0][2]) as hr:\n        channels = min(lr.count, hr.count)\n        descriptions = (lr.descriptions, hr.descriptions)\n        dtypes = (lr.dtypes[0], hr.dtypes[0])\n    nir_available = channels >= 4 and all(\n        name and "nir" in name.casefold() for name in (descriptions[0][3], descriptions[1][3])\n    )\n    return {"rows": inventory, "identity": manifest.hexdigest(), "channels": channels,\n            "nir_available": nir_available, "descriptions": descriptions, "dtypes": dtypes,\n            "counts": {"official_train": len(inventory["train"]),\n                       "official_test": len(inventory["test"])},\n            "seed": seed, "validation_percent": val_percent}\n\n\ndef oli_split(inventory: dict, split: str) -> list[tuple[str, Path, Path]]:\n    if split == "test":\n        return inventory["rows"]["test"]\n    if split not in {"train", "val"}:\n        raise ValueError(split)\n    result = []\n    for item in inventory["rows"]["train"]:\n        value = hashlib.sha256(f"{inventory[\'seed\']}:{item[0]}".encode()).digest()\n        assigned = "val" if int.from_bytes(value[:4], "big") % 100 < inventory["validation_percent"] else "train"\n        if assigned == split:\n            result.append(item)\n    return result\n\n\ndef decode(image: np.ndarray, dtype: str, radiometry: str) -> np.ndarray:\n    if np.dtype(dtype) == np.dtype("uint8"):\n        image = image.astype(np.float32) / 255.0\n    elif np.issubdtype(np.dtype(dtype), np.integer):\n        image = image.astype(np.float32) / 10000.0\n    else:\n        image = image.astype(np.float32)\n        if np.nanmax(image) > 1.5:\n            raise ValueError("Floating OLI2MSI reflectance exceeds 1.5; inspect source scaling")\n    image = np.clip(image, 0, 1)\n    if radiometry == "quantized8":\n        image = np.rint(image * 255) / 255\n    elif radiometry != "reflectance16":\n        raise ValueError(radiometry)\n    return image\n\n\nclass OliPairs(Dataset):\n    def __init__(self, inventory: dict, split: str, modality: str,\n                 radiometry: str, augment: bool = False):\n        self.rows = oli_split(inventory, split)\n        self.modality, self.radiometry, self.augment = modality, radiometry, augment\n        if not self.rows:\n            raise ValueError(f"Empty OLI2MSI {split} split")\n        if modality != "rgb" and not inventory["nir_available"]:\n            raise ValueError("OLI2MSI lacks verified paired NIR; RGB+NIR experiments cannot run")\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, index: int):\n        name, lr_path, hr_path = self.rows[index]\n        bands = [1, 2, 3] if self.modality == "rgb" else [1, 2, 3, 4]\n        arrays = []\n        for path, size in ((lr_path, 160), (hr_path, 480)):\n            with rasterio.open(path) as src:\n                if src.width != size or src.height != size:\n                    raise ValueError(f"Expected {size}x{size}: {path}")\n                if src.count < len(bands):\n                    raise ValueError(f"Missing bands in {path}")\n                raw = src.read(bands)\n                image = decode(raw, src.dtypes[0], self.radiometry)\n                mask = np.all(src.read_masks(bands) > 0, axis=0) & np.isfinite(raw).all(axis=0)\n                arrays.append((image, mask))\n        (lr, lr_mask), (hr, hr_mask) = arrays\n        mask = hr_mask & np.repeat(np.repeat(lr_mask, 3, 0), 3, 1)\n        if not mask.any():\n            raise ValueError(f"No jointly valid pixels: {name}")\n        payload = [lr, hr, mask[None].astype(np.float32)]\n        if self.augment:\n            rotation = int(torch.randint(4, ()).item())\n            payload = [np.rot90(x, rotation, axes=(-2, -1)) for x in payload]\n            if torch.rand(()) < 0.5:\n                payload = [x[..., ::-1] for x in payload]\n        return (*[torch.from_numpy(x.copy()) for x in payload], name)\n\n\nclass StudyPairs(Dataset):\n    def __init__(self, domain: str, root: Path, split: str, modality: str,\n                 radiometry: str, inventory: dict | None = None, augment: bool = False):\n        self.modality = modality\n        channels = "rgb" if modality == "rgb" else "rgbnir"\n        if domain == "oli2msi":\n            self.source = OliPairs(inventory or oli_inventory(root), split, channels, radiometry, augment)\n        elif domain == "mahimandal":\n            self.source = MahimandalPairs(root, split, channels, radiometry, augment, 160)\n        else:\n            raise ValueError(domain)\n\n    def __len__(self) -> int:\n        return len(self.source)\n\n    def __getitem__(self, index: int):\n        lr, hr, mask, row = self.source[index]\n        if self.modality == "rgbnir_rgb":\n            hr = hr[:3]\n        pair_id = row if isinstance(row, str) else self.source.rows[index]["pair_id"]\n        return lr, hr, mask, pair_id\n\n\ndef load_model(kind: str, name: str, sources: Path, modality: str,\n               architecture: Path | None, width: int):\n    if kind == "benchmark":\n        model, spec = load_benchmark(name, sources, modality)\n        return model, spec\n    raise ValueError(kind)\n\n\ndef predict(kind: str, model, spec: dict, lr: torch.Tensor):\n    if model is None:\n        return F.interpolate(lr, scale_factor=3, mode="bicubic", align_corners=False)\n    if kind == "benchmark":\n        return predict_benchmark(model, spec, lr)\n    raise ValueError(kind)\n\n\ndef protocol(args, identity: str, spec: dict) -> dict:\n    value = {"source_dataset": "oli2msi", "dataset_identity": identity,\n            "model_kind": args.kind, "model_name": args.model, "model_spec": spec,\n            "modality": args.modality, "radiometry": args.radiometry,\n            "seed": args.seed, "batch_size": args.batch_size, "lr": args.lr,\n            "optimizer": "Adam", "loss": (\n                "masked L1 + RGB LPIPS + logistic GAN after warmup"\n                if args.kind == "benchmark" and args.model == "RealESRGAN_SRVGG_X3"\n                else "masked L1"), "augmentation": "D4",\n            "train_geometry": [160, 480], "selection": "full-frame OLI2MSI validation PSNR"}\n    if args.kind == "benchmark" and args.model == "RealESRGAN_SRVGG_X3":\n        value["gan_warmup_epochs"] = max(1, args.epochs // 3)\n    return json.loads(json.dumps(value, allow_nan=False))\n\n\ndef training_root(args) -> Path:\n    return args.output / "trained_oli2msi" / args.kind / args.radiometry / args.modality / args.model / f"seed_{args.seed}"\n\n\ndef train(args, inventory: dict) -> Path:\n    root = training_root(args)\n    root.mkdir(parents=True, exist_ok=True)\n    torch.manual_seed(args.seed)\n    np.random.seed(args.seed)\n    random.seed(args.seed)\n    model, spec = load_model(args.kind, args.model, args.sources, args.modality,\n                             args.architecture, args.width)\n    if model is None:\n        raise ValueError("Bicubic is an evaluation baseline")\n    signature = protocol(args, inventory["identity"], spec)\n    lock = root / "protocol.json"\n    if lock.is_file() and json.loads(lock.read_text()) != signature:\n        raise ValueError(f"Existing training protocol differs: {lock}")\n    write_json(lock, signature)\n    device = torch.device(args.device)\n    model.to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)\n    use_gan = args.kind == "benchmark" and args.model == "RealESRGAN_SRVGG_X3"\n    discriminator = discriminator_optimizer = perceptual = None\n    if use_gan:\n        import lpips\n        discriminator = load_discriminator(args.sources).to(device)\n        discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=args.lr)\n        perceptual = lpips.LPIPS(net="alex").to(device).eval()\n        for parameter in perceptual.parameters():\n            parameter.requires_grad_(False)\n    training = StudyPairs("oli2msi", args.oli, "train", args.modality, args.radiometry, inventory, True)\n    validation = StudyPairs("oli2msi", args.oli, "val", args.modality, args.radiometry, inventory)\n    last, best = root / "last.pt", root / "best.pt"\n    epoch0, best_psnr, stale = 0, -float("inf"), 0\n    if last.is_file():\n        state = torch.load(last, map_location="cpu", weights_only=False)\n        if state["protocol"] != signature:\n            raise ValueError("Checkpoint protocol differs from current request")\n        model.load_state_dict(state["model"], strict=True)\n        optimizer.load_state_dict(state["optimizer"])\n        if use_gan:\n            discriminator.load_state_dict(state["discriminator"], strict=True)\n            discriminator_optimizer.load_state_dict(state["discriminator_optimizer"])\n        epoch0, best_psnr, stale = state["epoch"], state["best_psnr"], state["stale"]\n    for epoch in range(epoch0 + 1, args.epochs + 1):\n        if stale >= args.patience:\n            break\n        torch.manual_seed(args.seed + epoch)\n        loader = DataLoader(training, batch_size=args.batch_size, shuffle=True,\n                            num_workers=args.workers, pin_memory=device.type == "cuda")\n        model.train()\n        losses = []\n        for lr, hr, mask, _ in loader:\n            lr, hr, mask = lr.to(device), hr.to(device), mask.to(device)\n            output = predict(args.kind, model, spec, lr)\n            loss = mae(output, hr, mask)\n            if use_gan and epoch > max(1, args.epochs // 3):\n                discriminator_optimizer.zero_grad(set_to_none=True)\n                real_logits = discriminator(hr[:, :3])\n                fake_logits = discriminator(output[:, :3].detach())\n                d_loss = F.softplus(-real_logits).mean() + F.softplus(fake_logits).mean()\n                d_loss.backward()\n                discriminator_optimizer.step()\n                for parameter in discriminator.parameters():\n                    parameter.requires_grad_(False)\n                loss = (loss + 0.1 * perceptual(output[:, :3] * 2 - 1,\n                                                 hr[:, :3] * 2 - 1).mean()\n                        + 0.005 * F.softplus(-discriminator(output[:, :3])).mean())\n                for parameter in discriminator.parameters():\n                    parameter.requires_grad_(True)\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)\n            optimizer.step()\n            losses.append(float(loss.detach()))\n        model.eval()\n        scores = []\n        with torch.no_grad():\n            for lr, hr, mask, _ in DataLoader(validation, batch_size=1):\n                output = predict(args.kind, model, spec, lr.to(device)).float().clamp(0, 1).cpu()\n                scores.append(float(psnr(output, hr, mask)))\n        score = float(np.mean(scores))\n        improved = score > best_psnr\n        stale = 0 if improved else stale + 1\n        best_psnr = max(best_psnr, score)\n        state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),\n                 "epoch": epoch, "best_psnr": best_psnr, "stale": stale,\n                 "protocol": signature}\n        if use_gan:\n            state["discriminator"] = discriminator.state_dict()\n            state["discriminator_optimizer"] = discriminator_optimizer.state_dict()\n        temporary = root / "checkpoint.tmp"\n        torch.save(state, temporary)\n        temporary.replace(last)\n        if improved:\n            torch.save(state, temporary)\n            temporary.replace(best)\n        print(f"{args.kind}/{args.model}/{args.modality}: epoch {epoch}/{args.epochs} "\n              f"train_l1={np.mean(losses):.6f} val_psnr={score:.4f} stale={stale}/{args.patience}", flush=True)\n    if not best.is_file():\n        raise FileNotFoundError(f"No best checkpoint: {best}")\n    return best\n\n\n@torch.no_grad()\ndef evaluate(args, inventory: dict, checkpoint: Path, domain: str) -> dict:\n    model, spec = load_model(args.kind, args.model, args.sources, args.modality,\n                             args.architecture, args.width)\n    if model is not None:\n        state = torch.load(checkpoint, map_location="cpu", weights_only=False)\n        if state["protocol"].get("modality") != args.modality:\n            raise ValueError("Checkpoint modality does not match request")\n        if state["protocol"].get("radiometry") != args.radiometry:\n            raise ValueError("Checkpoint radiometry does not match request")\n        if args.kind == "benchmark" and state["protocol"].get("model", {}).get("name") != args.model:\n            raise ValueError("Checkpoint benchmark architecture does not match request")\n        model.load_state_dict(state["model"], strict=True)\n    checkpoint_hash = digest(checkpoint) if model is not None else "bicubic"\n    target_root = args.oli if domain == "oli2msi" else args.mahimandal\n    target_identity = inventory["identity"] if domain == "oli2msi" else digest(target_root / "paper_release_manifest.csv")\n    destination = args.output / "evaluation" / args.trained_on / domain / args.kind / args.radiometry / args.modality / args.model / checkpoint_hash[:12]\n    receipt = {"trained_on": args.trained_on, "tested_on": domain,\n               "source_checkpoint_sha256": checkpoint_hash, "target_identity": target_identity,\n               "modality": args.modality, "radiometry": args.radiometry,\n               "geometry": [160, 480], "metric_domain": "reflectance [0,1]",\n               "border_crop": args.border_crop}\n    lock = destination / "protocol.json"\n    if lock.is_file() and json.loads(lock.read_text()) != receipt:\n        raise ValueError(f"Existing evaluation differs: {lock}")\n    write_json(lock, receipt)\n    result = destination / "summary.json"\n    if result.is_file() and (destination / "per_image.csv").is_file():\n        print("Reused:", result, flush=True)\n        return json.loads(result.read_text())\n    device = torch.device(args.device)\n    if model is not None:\n        model.to(device).eval()\n    dataset = StudyPairs(domain, target_root, "test", args.modality,\n                         args.radiometry, inventory if domain == "oli2msi" else None)\n    rows = []\n    pictured_classes = set()\n    for index, (lr, hr, mask, pair_id) in enumerate(DataLoader(dataset, batch_size=1)):\n        if device.type == "cuda":\n            torch.cuda.synchronize()\n        started = time.perf_counter()\n        output = predict(args.kind, model, spec, lr.to(device)).float().clamp(0, 1)\n        if device.type == "cuda":\n            torch.cuda.synchronize()\n        elapsed = (time.perf_counter() - started) * 1000\n        if tuple(output.shape) != tuple(hr.shape) or not torch.isfinite(output).all():\n            raise ValueError(f"Invalid prediction for {pair_id[0]}")\n        measures = metric_bundle(output, hr.to(device), mask.to(device), border=args.border_crop)\n        metadata = dataset.source.rows[index] if domain == "mahimandal" else {}\n        scene = metadata.get("landscape_class", "unlabeled")\n        rows.append({"pair_id": str(pair_id[0]),\n                     "landscape_class": scene,\n                     "geographic_stratum": metadata.get("geographic_stratum", "unlabeled"),\n                     "spatial_block_id": metadata.get("spatial_block_id", ""),\n                     "day_gap": metadata.get("day_gap", ""),\n                     "inference_ms": elapsed, **measures})\n        first_of_class = domain == "mahimandal" and scene not in pictured_classes\n        if index < 5 or first_of_class:\n            import matplotlib.pyplot as plt\n            name = (f"class_{\'\'.join(c if c.isalnum() else \'_\' for c in scene)}.png"\n                    if first_of_class else f"test_{index:04d}.png")\n            visual = destination / "figures" / name\n            visual.parent.mkdir(parents=True, exist_ok=True)\n            lr_rgb = F.interpolate(lr[:, :3], scale_factor=3, mode="nearest")[0].numpy()\n            target_rgb = hr[0, :3].numpy()\n            prediction_rgb = output[0, :3].cpu().numpy()\n            panels = (lr_rgb, target_rgb, prediction_rgb,\n                      np.abs(prediction_rgb - target_rgb))\n            figure, axes = plt.subplots(1, 4, figsize=(16, 4))\n            for axis, image, title in zip(axes, panels,\n                                          ("Landsat LR", "Sentinel HR", args.model, "RGB absolute error")):\n                shown = image.transpose(1, 2, 0)\n                axis.imshow(np.clip(shown / 0.3, 0, 1) if title != "RGB absolute error"\n                            else np.clip(shown / 0.1, 0, 1))\n                axis.set_title(title)\n                axis.axis("off")\n            figure.suptitle(f"{domain} test/{index}: {pair_id[0]} | {scene} | display stretch only")\n            figure.tight_layout()\n            figure.savefig(visual, dpi=140)\n            plt.close(figure)\n            pictured_classes.add(scene)\n    write_csv(destination / "per_image.csv", rows)\n    summary = {"model": args.model, "kind": args.kind, "modality": args.modality,\n               "radiometry": args.radiometry, "trained_on": args.trained_on,\n               "tested_on": domain, "count": len(rows), "parameters": spec["parameters"],\n               "checkpoint_sha256": checkpoint_hash,\n               **{key: float(np.mean([r[key] for r in rows])) for key in rows[0]\n                  if isinstance(rows[0][key], (int, float))}}\n    write_json(result, summary)\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=("audit", "train", "evaluate"))\n    parser.add_argument("--oli", type=Path, required=True)\n    parser.add_argument("--mahimandal", type=Path)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--sources", type=Path)\n    parser.add_argument("--kind", choices=("benchmark",), default="benchmark")\n    parser.add_argument("--model", default="bicubic")\n    parser.add_argument("--architecture", type=Path)\n    parser.add_argument("--width", type=int, default=48)\n    parser.add_argument("--modality", choices=tuple(PROFILES), default="rgb")\n    parser.add_argument("--radiometry", choices=("reflectance16", "quantized8"), default="reflectance16")\n    parser.add_argument("--trained-on", choices=("mahimandal", "oli2msi"), default="mahimandal")\n    parser.add_argument("--tested-on", choices=("mahimandal", "oli2msi"), default="oli2msi")\n    parser.add_argument("--checkpoint", type=Path)\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--patience", type=int, default=3)\n    parser.add_argument("--batch-size", type=int, default=1)\n    parser.add_argument("--lr", type=float, default=1e-4)\n    parser.add_argument("--workers", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--border-crop", type=int, default=3)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    args = parser.parse_args()\n    inventory = oli_inventory(args.oli, args.seed)\n    if args.action == "audit":\n        result = {key: value for key, value in inventory.items() if key != "rows"}\n        result["splits"] = {name: len(oli_split(inventory, name)) for name in ("train", "val", "test")}\n        result["geometry"] = [160, 480]\n        write_json(args.output / "oli_audit.json", result)\n        print(json.dumps(result, indent=2), flush=True)\n        return\n    if args.action == "train":\n        print(train(args, inventory))\n        return\n    if args.tested_on == "mahimandal" and args.mahimandal is None:\n        parser.error("--mahimandal is required for that test domain")\n    checkpoint = args.checkpoint or training_root(args) / "best.pt"\n    evaluate(args, inventory, checkpoint, args.tested_on)\n\n\nif __name__ == "__main__":\n    main()\n'
runner_path = CODE_DIR / 'cross_dataset_models.py'
if runner_path.is_file() and runner_path.read_text(encoding='utf-8') != RUNNER_SOURCE:
    raise ValueError('Transfer code changed after this run was created; use a new cross-study directory')
runner_path.write_text(RUNNER_SOURCE, encoding='utf-8')
from cross_dataset_models import oli_inventory, oli_split
inventory = oli_inventory(OLI_ROOT)
counts = {split: len(oli_split(inventory, split)) for split in ('train','val','test')}
if inventory['counts'] != {'official_train': 5225, 'official_test': 100}:
    raise ValueError(f'Unexpected official OLI2MSI counts: {inventory["counts"]}')
print('OLI bands:', inventory['channels'], inventory['descriptions'])
print('Paired NIR verified:', inventory['nir_available'])
print('Study splits:', counts)
split_rows = [{'pair_id': pair_id, 'split': split}
              for split in ('train', 'val', 'test')
              for pair_id, _, _ in oli_split(inventory, split)]
(CROSS_HOME / 'results').mkdir(exist_ok=True)
pd.DataFrame(split_rows).to_csv(
    CROSS_HOME / 'results' / 'oli_split_manifest.csv', index=False
)
run([PYTHON, runner_path, 'audit', '--oli', OLI_ROOT,
     '--output', CROSS_HOME / 'results'])


## 5. Evaluate Mahimandal-trained official benchmark models on OLI2MSI


In [ ]:
RUNNER = CODE_DIR / 'cross_dataset_models.py'
RESULTS = CROSS_HOME / 'results' / 'notebook3_baselines'
RESULTS.mkdir(parents=True, exist_ok=True)
COMMON = [PYTHON, RUNNER, '--oli', OLI_ROOT, '--mahimandal', MAH_ROOT,
          '--output', RESULTS, '--sources', BENCH_SOURCES,
          '--device', 'cuda', '--border-crop', 3]

preflight = json.loads((BENCH_RESULTS / 'preflight_status.json').read_text())
trained = []
published_models = {'EDSR', 'RCAN', 'RDN', 'CAFRN', 'DCM', 'FENet', 'RealESRGAN_SRVGG_X3'}
passed_models = {key.split('/')[1] for key, value in preflight.items()
                 if key.startswith('rgb/') and value['status'] == 'passed'}
if not published_models <= passed_models:
    raise ValueError(f'Use the original preflight receipt; missing: {sorted(published_models - passed_models)}')
for radiometry in ('reflectance16',):
    for model in sorted(published_models):
        checkpoint = BENCH_RESULTS / radiometry / 'rgb' / model / 'seed_42' / 'best.pt'
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        trained.append((radiometry, model))
        run(COMMON + ['evaluate', '--kind', 'benchmark', '--model', model,
             '--modality', 'rgb', '--radiometry', radiometry,
             '--trained-on', 'mahimandal', '--tested-on', 'oli2msi',
             '--checkpoint', checkpoint])
    run(COMMON + ['evaluate', '--kind', 'benchmark', '--model', 'bicubic',
         '--modality', 'rgb', '--radiometry', radiometry,
         '--trained-on', 'mahimandal', '--tested-on', 'oli2msi'])
if not trained:
    raise RuntimeError('No trained RGB benchmark checkpoints were found')
print('Benchmark checkpoints evaluated:', trained)


## 7. Export the cross-dataset test report


In [ ]:
paths = sorted(RESULTS.glob('evaluation/*/*/*/*/*/*/*/summary.json'))
if not paths:
    raise RuntimeError('No completed evaluations were found')
summary = pd.DataFrame([json.loads(p.read_text()) for p in paths])
summary.to_csv(RESULTS / 'transfer_summary.csv', index=False)
display(summary.sort_values(['tested_on', 'trained_on', 'psnr'], ascending=[True, True, False]))
detailed = []
for path in paths:
    identity = json.loads(path.read_text())
    frame = pd.read_csv(path.parent / 'per_image.csv')
    for field in ('trained_on', 'tested_on', 'kind', 'radiometry', 'modality', 'model'):
        frame[field] = identity[field]
    detailed.append(frame)
per_image = pd.concat(detailed, ignore_index=True)
per_image.to_csv(RESULTS / 'all_test_per_image.csv', index=False)
metrics = [key for key in ('psnr', 'ssim', 'mae', 'rmse', 'edge_f1',
           'sam_degrees', 'ergas', 'uiqi', 'scc', 'rgb_psnr', 'y_psnr',
           'ndvi_rmse', 'ndvi_correlation', 'inference_ms') if key in per_image]
dimensions = ['trained_on', 'tested_on', 'kind', 'radiometry', 'modality', 'model']
grouped = per_image.groupby(dimensions, dropna=False)[metrics].agg(['mean', 'std', 'count'])
grouped.columns = ['_'.join(column) for column in grouped.columns]
grouped.reset_index().to_csv(RESULTS / 'metrics_mean_std_count.csv', index=False)
for field, filename in (('landscape_class', 'by_landscape_class.csv'),
                        ('geographic_stratum', 'by_geographic_stratum.csv')):
    if field not in per_image:
        continue
    class_table = (per_image[per_image.tested_on == 'mahimandal']
        .groupby(dimensions + [field], dropna=False)[metrics]
        .agg(['mean', 'std', 'count']))
    if not class_table.empty:
        class_table.columns = ['_'.join(column) for column in class_table.columns]
        class_table = class_table.reset_index()
        class_table.to_csv(RESULTS / filename, index=False)
        display(class_table[[*dimensions, field, 'psnr_mean', 'psnr_std',
                             'psnr_count', 'ssim_mean', 'edge_f1_mean']].round(4))
extremes = []
for key, frame in per_image.groupby(dimensions, dropna=False):
    for label, row in (('lowest_psnr', frame.loc[frame.psnr.idxmin()]),
                       ('highest_psnr', frame.loc[frame.psnr.idxmax()])):
        extremes.append(dict(zip(dimensions, key), case=label,
                             pair_id=row.pair_id, landscape_class=row.get('landscape_class', ''),
                             psnr=float(row.psnr), ssim=float(row.ssim),
                             edge_f1=float(row.edge_f1)))
pd.DataFrame(extremes).to_csv(RESULTS / 'test_extremes.csv', index=False)
import numpy as np
import matplotlib.pyplot as plt
for (target, radiometry), group in summary.groupby(['tested_on', 'radiometry']):
    ordered = group.sort_values('psnr')
    figure, axis = plt.subplots(figsize=(max(7, len(ordered) * .75), 4.5))
    axis.barh(ordered.model, ordered.psnr, color='#287e83')
    axis.set(xlabel='Mean PSNR (dB), reflectance [0,1]',
             title=f'{target} official test | {radiometry} | {len(ordered)} models')
    axis.grid(axis='x', alpha=.2)
    figure.tight_layout()
    figure.savefig(RESULTS / f'psnr_{target}_{radiometry}.png', dpi=180)
    plt.close(figure)
paired = []
for target, group in summary.groupby('tested_on'):
    for radiometry, subset in group.groupby('radiometry'):
        baseline_row = subset[subset.model == 'bicubic']
        if baseline_row.empty:
            continue
        baseline_file = next(p.parent / 'per_image.csv' for p in paths
                             if json.loads(p.read_text())['tested_on'] == target
                             and json.loads(p.read_text())['radiometry'] == radiometry
                             and json.loads(p.read_text())['model'] == 'bicubic')
        baseline = pd.read_csv(baseline_file).set_index('pair_id')
        for path in paths:
            model_row = json.loads(path.read_text())
            if (model_row['tested_on'] != target or model_row['radiometry'] != radiometry
                    or model_row['model'] == 'bicubic'):
                continue
            candidate = pd.read_csv(path.parent / 'per_image.csv').set_index('pair_id')
            if set(candidate.index) != set(baseline.index):
                raise ValueError('Per-image test IDs differ from bicubic')
            delta = (candidate.loc[baseline.index, 'psnr'] - baseline.psnr).to_numpy()
            rng = np.random.default_rng(42)
            draws = rng.choice(delta, size=(2000, len(delta)), replace=True).mean(axis=1)
            paired.append({'trained_on': model_row['trained_on'], 'tested_on': target,
                'radiometry': radiometry, 'model': model_row['model'], 'count': len(delta),
                'mean_psnr_delta_vs_bicubic': float(delta.mean()),
                'fraction_beating_bicubic': float(np.mean(delta > 0)),
                'ci95_low': float(np.quantile(draws, .025)),
                'ci95_high': float(np.quantile(draws, .975))})
if paired:
    comparison = pd.DataFrame(paired)
    comparison.to_csv(RESULTS / 'paired_vs_bicubic.csv', index=False)
    display(comparison)
from IPython.display import Image
available_galleries = sorted(RESULTS.glob(
    'evaluation/*/mahimandal/*/*/*/*/*/figures/class_*.png'))
seen_classes, galleries = set(), []
for figure_path in available_galleries:
    if figure_path.stem not in seen_classes:
        galleries.append(figure_path)
        seen_classes.add(figure_path.stem)
galleries += sorted(RESULTS.glob(
    'evaluation/*/oli2msi/*/*/*/*/*/figures/test_0000.png'))[:1]
print('Displayed classes:', sorted(seen_classes))
for figure_path in galleries:
    print(figure_path.relative_to(RESULTS))
    display(Image(filename=str(figure_path)))
import zipfile
bundle = CROSS_HOME / (RESULTS.name + '_paper_results.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.rglob('*')):
        if path.is_file() and path.suffix in {'.json', '.csv', '.png', '.txt'}:
            archive.write(path, path.relative_to(RESULTS))
    for name in ('oli_audit.json', 'oli_split_manifest.csv'):
        archive.write(CROSS_HOME / 'results' / name, 'protocol/' + name)
    archive.write(RUNNER, 'protocol/cross_dataset_models.py')
print('Results bundle:', bundle)
display(FileLink(str(bundle)))
